# NFL API Exploration with Python
This notebook walks through how to:
1. Discover and explore NFL API endpoints
2. Ingest data from those endpoints
3. Load the data into pandas DataFrames
4. Join multiple tables together
5. Run basic analysis

**API used:** ESPN's public NFL API (no API key required)  
**Base URL:** `https://site.api.espn.com/apis/site/v2/sports/football/nfl/`

## 1. Setup & Imports

In [ ]:
import requests
import pandas as pd
import json
from pprint import pprint

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

BASE_URL = 'https://site.api.espn.com/apis/site/v2/sports/football/nfl'
print('Setup complete.')

## 2. How to Explore an API Endpoint

The first step is always to **inspect the raw response** — understand its structure before writing any parsing logic.  
We'll write a helper that fetches a URL and pretty-prints the JSON so you can see exactly what keys exist.

In [ ]:
def fetch(endpoint: str, params: dict = None) -> dict:
    """Fetch an endpoint and return parsed JSON, or raise on failure."""
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()  # raises if status >= 400
    return response.json()


def explore(endpoint: str, params: dict = None, depth: int = 1):
    """
    Print the top-level keys of an API response (and one level deeper).
    Use this to understand what an endpoint returns before parsing it.
    """
    data = fetch(endpoint, params)
    print(f"\nEndpoint: {BASE_URL}/{endpoint}")
    print(f"Top-level keys: {list(data.keys())}")
    print()
    for key, value in data.items():
        if isinstance(value, dict):
            print(f"  [{key}] → dict with keys: {list(value.keys())}")
        elif isinstance(value, list):
            print(f"  [{key}] → list of {len(value)} items")
            if value and isinstance(value[0], dict):
                print(f"           first item keys: {list(value[0].keys())}")
        else:
            print(f"  [{key}] → {repr(value)[:80]}")
    return data

print('Helper functions defined.')

### 2a. Explore: Scoreboard (current week's games)

In [ ]:
scoreboard_raw = explore('scoreboard')

### 2b. Explore: Teams

In [ ]:
teams_raw = explore('teams')

### 2c. Explore: Standings

In [ ]:
standings_raw = explore('standings')

### 2d. Available Endpoints (cheat sheet)

| Endpoint | What it returns |
|---|---|
| `scoreboard` | Games for current/selected week |
| `scoreboard?dates=20241110` | Games on a specific date |
| `teams` | All 32 NFL teams |
| `teams/{id}` | One team's details |
| `teams/{id}/roster` | Players on a team |
| `standings` | Division standings |
| `athletes/{id}` | Single player details |
| `summary?event={id}` | Box score for one game |

## 3. Ingest Data into DataFrames

Now that we know the shape of the data, we'll parse each endpoint into a clean DataFrame.

### 3a. Teams DataFrame

In [ ]:
def get_teams() -> pd.DataFrame:
    data = fetch('teams', params={'limit': 32})
    teams = data['sports'][0]['leagues'][0]['teams']

    rows = []
    for entry in teams:
        t = entry['team']
        rows.append({
            'team_id':       t['id'],
            'abbreviation':  t['abbreviation'],
            'display_name':  t['displayName'],
            'short_name':    t['shortDisplayName'],
            'location':      t['location'],
            'nickname':      t['name'],
            'color':         t.get('color', ''),
            'logo_url':      t['logos'][0]['href'] if t.get('logos') else '',
        })

    return pd.DataFrame(rows)


df_teams = get_teams()
print(f"Teams loaded: {len(df_teams)} rows")
df_teams.head()

### 3b. Standings DataFrame

In [ ]:
def get_standings() -> pd.DataFrame:
    data = fetch('standings')
    conferences = data['children']

    rows = []
    for conf in conferences:
        conf_name = conf['name']  # AFC / NFC
        for division in conf['children']:
            div_name = division['name']
            for entry in division['standings']['entries']:
                team_id   = entry['team']['id']
                team_abbr = entry['team']['abbreviation']
                stats = {s['name']: s['value'] for s in entry['stats']}
                rows.append({
                    'team_id':     team_id,
                    'team_abbr':   team_abbr,
                    'conference':  conf_name,
                    'division':    div_name,
                    'wins':        stats.get('wins', 0),
                    'losses':      stats.get('losses', 0),
                    'ties':        stats.get('ties', 0),
                    'win_pct':     stats.get('winPercent', 0),
                    'points_for':  stats.get('pointsFor', 0),
                    'points_against': stats.get('pointsAgainst', 0),
                    'streak':      stats.get('streak', ''),
                })

    return pd.DataFrame(rows)


df_standings = get_standings()
print(f"Standings loaded: {len(df_standings)} rows")
df_standings.head(10)

### 3c. Roster DataFrame (one team — show pattern, then loop all)

In [ ]:
def get_roster(team_id: str) -> pd.DataFrame:
    data = fetch(f'teams/{team_id}/roster')
    athletes = data.get('athletes', [])

    rows = []
    for position_group in athletes:
        position = position_group.get('position', '')
        for player in position_group.get('items', []):
            rows.append({
                'player_id':    player['id'],
                'team_id':      str(team_id),
                'full_name':    player.get('fullName', ''),
                'position':     player.get('position', {}).get('abbreviation', ''),
                'jersey':       player.get('jersey', ''),
                'age':          player.get('age', None),
                'experience':   player.get('experience', {}).get('years', None),
                'college':      player.get('college', {}).get('name', '') if player.get('college') else '',
            })
    return pd.DataFrame(rows)


# Demo: pull roster for the Kansas City Chiefs (team_id = 12)
chiefs_id = df_teams.loc[df_teams['abbreviation'] == 'KC', 'team_id'].values[0]
df_chiefs_roster = get_roster(chiefs_id)
print(f"Chiefs roster: {len(df_chiefs_roster)} players")
df_chiefs_roster.head(10)

In [ ]:
# Load rosters for ALL 32 teams (takes ~30 seconds)
# Comment this out if you just want to demo with one team.
import time

all_rosters = []
for _, row in df_teams.iterrows():
    try:
        roster = get_roster(row['team_id'])
        roster['team_abbr'] = row['abbreviation']
        all_rosters.append(roster)
        time.sleep(0.2)  # be polite to the API
    except Exception as e:
        print(f"  Skipped {row['abbreviation']}: {e}")

df_rosters = pd.concat(all_rosters, ignore_index=True)
print(f"All rosters loaded: {len(df_rosters)} players across {df_rosters['team_id'].nunique()} teams")
df_rosters.head()

## 4. Joining Tables

Now we have three DataFrames:
- `df_teams` — team metadata (name, location, color)
- `df_standings` — wins/losses/points
- `df_rosters` — all players

We'll join them on `team_id` to answer richer questions.

### 4a. Teams + Standings (inner join)

In [ ]:
# Both tables share 'team_id'
df_teams_standings = pd.merge(
    df_teams,
    df_standings,
    on='team_id',
    how='inner'
)

# Drop duplicate abbreviation column from standings
df_teams_standings.drop(columns=['team_abbr'], inplace=True)

print(f"Merged shape: {df_teams_standings.shape}")
df_teams_standings[['display_name', 'conference', 'division', 'wins', 'losses', 'win_pct', 'points_for']].head(10)

### 4b. Rosters + Team Info (add city/nickname to every player)

In [ ]:
df_rosters_full = pd.merge(
    df_rosters,
    df_teams[['team_id', 'display_name', 'location', 'nickname']],
    on='team_id',
    how='left'
)

print(f"Players with team info: {len(df_rosters_full)}")
df_rosters_full[['full_name', 'position', 'jersey', 'age', 'display_name']].head(10)

### 4c. Rosters + Standings (what division/record does each player's team have?)

In [ ]:
df_players_standings = pd.merge(
    df_rosters_full,
    df_standings[['team_id', 'conference', 'division', 'wins', 'losses', 'win_pct']],
    on='team_id',
    how='left'
)

print(f"Shape: {df_players_standings.shape}")
df_players_standings[['full_name', 'position', 'display_name', 'conference', 'wins', 'losses']].head(10)

## 5. Basic Analysis

### 5a. Top 10 teams by win percentage

In [ ]:
top_teams = (
    df_teams_standings
    .sort_values('win_pct', ascending=False)
    [['display_name', 'conference', 'division', 'wins', 'losses', 'win_pct', 'points_for', 'points_against']]
    .reset_index(drop=True)
)
top_teams.head(10)

### 5b. Average win % by conference

In [ ]:
df_teams_standings.groupby('conference').agg(
    avg_wins=('wins', 'mean'),
    avg_losses=('losses', 'mean'),
    avg_win_pct=('win_pct', 'mean'),
    avg_pts_for=('points_for', 'mean'),
    avg_pts_against=('points_against', 'mean'),
).round(2)

### 5c. Average win % by division

In [ ]:
df_teams_standings.groupby(['conference', 'division']).agg(
    teams=('display_name', 'count'),
    avg_wins=('wins', 'mean'),
    avg_win_pct=('win_pct', 'mean'),
).round(2).sort_values('avg_wins', ascending=False)

### 5d. Roster size by team

In [ ]:
roster_size = (
    df_rosters_full
    .groupby('display_name')
    .agg(roster_size=('player_id', 'count'),
         avg_age=('age', 'mean'),
         avg_experience=('experience', 'mean'))
    .round(1)
    .sort_values('roster_size', ascending=False)
)
roster_size.head(10)

### 5e. Position distribution across the league

In [ ]:
position_counts = (
    df_rosters_full['position']
    .value_counts()
    .reset_index()
)
position_counts.columns = ['position', 'count']
position_counts.head(15)

### 5f. Do better teams have older/more experienced rosters?

In [ ]:
team_experience = (
    df_players_standings
    .groupby(['display_name', 'wins', 'win_pct'])
    .agg(avg_age=('age', 'mean'),
         avg_exp=('experience', 'mean'))
    .round(2)
    .reset_index()
    .sort_values('win_pct', ascending=False)
)

print("Correlation between win % and avg roster experience:")
corr = team_experience[['win_pct', 'avg_age', 'avg_exp']].corr()
print(corr.round(3))
print()
team_experience.head(10)

### 5g. Points scored vs points allowed (net point differential)

In [ ]:
df_teams_standings['point_diff'] = df_teams_standings['points_for'] - df_teams_standings['points_against']

(
    df_teams_standings[['display_name', 'wins', 'losses', 'points_for', 'points_against', 'point_diff']]
    .sort_values('point_diff', ascending=False)
    .reset_index(drop=True)
)

## 6. Saving to CSV (optional)

Export any DataFrame to a CSV for sharing or further analysis in Excel / Tableau.

In [ ]:
df_teams_standings.to_csv('nfl_teams_standings.csv', index=False)
df_rosters_full.to_csv('nfl_rosters.csv', index=False)
print('Files saved.')

## 7. Key Takeaways

| Concept | What we did |
|---|---|
| **Endpoint exploration** | Used `explore()` to print keys and structure before parsing |
| **Parameterized requests** | Passed `params={}` dicts to filter by date, limit, etc. |
| **DataFrame ingestion** | Looped over nested JSON and built flat row dicts |
| **Joining** | `pd.merge()` on shared `team_id` key — works like SQL JOIN |
| **Aggregation** | `groupby().agg()` for per-group stats |
| **Correlation** | `.corr()` to find relationships between numeric columns |

**Next steps to explore:**
- Pull `summary?event={game_id}` for play-by-play / box scores
- Use `scoreboard?dates=YYYYMMDD` to pull a full season of games
- Add `athletes/{id}` calls to get career stats per player